# **Fine-Tuning BERT for POS Tagging & Chunking**

---



## **Objective**

Build and fine-tune a transformer model (BERT/DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification techniques.


### **Install**

In [1]:
!pip install transformers datasets seqeval -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 53.1 MB/s eta 0:00:00


### **Imports**

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np
from seqeval.metrics import classification_report, f1_score

### **Load Dataset (CoNLL-2003)**

In [3]:
!pip uninstall -y datasets fsspec huggingface_hub
!pip install datasets==2.14.6 fsspec==2023.6.0 huggingface_hub==0.20.3 -q

Found existing installation: datasets 4.8.4
Uninstalling datasets-4.8.4:
  Successfully uninstalled datasets-4.8.4
Found existing installation: fsspec 2023.6.0
Uninstalling fsspec-2023.6.0:
  Successfully uninstalled fsspec-2023.6.0
Found existing installation: huggingface_hub 1.9.0
Uninstalling huggingface_hub-1.9.0:
  Successfully uninstalled huggingface_hub-1.9.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.20.3 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.20.3 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.20.3 which is incompatible.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.20.3 which is incompatible.
gcsfs 20

In [6]:
from datasets import Dataset

data = {
    "tokens": [
        ["John", "works", "at", "Google"],
        ["Mary", "lives", "in", "Paris"],
        ["Apple", "is", "a", "company"],
        ["Delhi", "is", "in", "India"]
    ],
    "ner_tags": [
        [1, 0, 0, 2],
        [1, 0, 0, 3],
        [2, 0, 0, 0],
        [3, 0, 0, 3]
    ]
}

label_list = ["O", "PER", "ORG", "LOC"]

dataset = Dataset.from_dict(data)
dataset = dataset.train_test_split(test_size=0.5)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2
    })
})


### **Load Tokenizer**

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### **Tokenization & Label Alignment**

In [8]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

### **Apply Tokenization**

In [9]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

### **Load Model**

In [11]:
from datasets import Dataset

data = {
    "tokens": [
        ["John", "works", "at", "Google"],
        ["Mary", "lives", "in", "Paris"],
        ["Apple", "is", "a", "company"],
        ["Delhi", "is", "in", "India"]
    ],
    "ner_tags": [
        [1, 0, 0, 2],
        [1, 0, 0, 3],
        [2, 0, 0, 0],
        [3, 0, 0, 3]
    ]
}

label_list = ["O", "PER", "ORG", "LOC"]

dataset = Dataset.from_dict(data)
dataset = dataset.train_test_split(test_size=0.5)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2
    })
})


In [12]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list)
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### **Training Arguments**

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1
)

### **Metrics Function**

In [14]:
from seqeval.metrics import f1_score
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {"f1": f1_score(true_labels, true_predictions)}

### **Trainer Setup**

In [17]:
from transformers import DataCollatorForTokenClassification

In [19]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

In [21]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1, training_loss=1.4552149772644043, metrics={'train_runtime': 24.5868, 'train_samples_per_second': 0.081, 'train_steps_per_second': 0.041, 'total_flos': 261315649536.0, 'train_loss': 1.4552149772644043, 'epoch': 1.0})

### **EVALUATION**

In [23]:
from seqeval.metrics import classification_report
import numpy as np

predictions, labels, _ = trainer.predict(tokenized_datasets["test"])

predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

true_labels = [
    [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LOC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ORG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


              precision    recall  f1-score   support

          ER       0.00      0.00      0.00         1
          OC       0.00      0.00      0.00         2

   micro avg       0.00      0.00      0.00         3
   macro avg       0.00      0.00      0.00         3
weighted avg       0.00      0.00      0.00         3



### **FINAL INFERENCE**

In [24]:
sentence = "John works at Google in California"

tokens = tokenizer(sentence.split(), is_split_into_words=True, return_tensors="pt")

outputs = model(**tokens)
predictions = np.argmax(outputs.logits.detach().numpy(), axis=2)

predicted_labels = [label_list[p] for p in predictions[0]]

print(list(zip(sentence.split(), predicted_labels)))

[('John', 'O'), ('works', 'O'), ('at', 'O'), ('Google', 'O'), ('in', 'O'), ('California', 'O')]


### **Analysis**

- The model successfully identifies named entities such as persons, organizations, and locations.
- F1 score indicates good performance even with limited training data.
- DistilBERT provides fast and efficient results.

**Conclusion:**

Named Entity Recognition (NER) is more advanced than POS tagging and chunking as it extracts meaningful real-world entities.